# CG Simulation Tutorial

This tutorial demonstrates how to run coarse-grained (CG) simulations of trp-cage using our pretrained MACE MFM 100K model.

---

Run the following command in the `transferable-cg` repository:

```bash
save_folder="save/folder" # path to save location
model_folder="model_weights/mace_models/mfm_100K/"

cd ..
mkdir -p ${save_folder}

uv run cg_sim "global_args.model_folder=./${model_folder}/"\
        "global_args.save_folder_name=${save_folder}/"\
        'global_args.save_freq=100'\
        'global_args.chk_freq=1000'\
        'global_args.num_data_points=10000'\
        'integrator_args.friction=1'\
        'integrator_args.dt=0.2'\
        'integrator_args.temperature=300'\
        'global_args.pdb_file=./docs/tutorials/trpcage/reference.pdb'\
        'global_args.start_positions=./docs/tutorials/trpcage/start_positions.npy'
```

This script performs **100 parallel simulations** of the trp-cage protein with the following configuration:

### Simulation Setup

| Parameter | Value | Description |
|-----------|-------|-------------|
| **Starting positions** | `start_positions.npy` | Array of shape `(100, n_atoms, 3)` - one starting configuration per replica |
| **Total steps** | 1,000,000 | `num_data_points × save_freq = 10,000 × 100` |
| **Saved frames** | 10,000 | One frame saved every 100 steps |
| **Checkpoints** | Every 1,000 frames | For resuming interrupted simulations |

### Integrator Settings

| Parameter | Value | Description |
|-----------|-------|-------------|
| **Timestep** | 0.2 fs | Integration timestep (`dt`) |
| **Temperature** | 300 K | Simulation temperature |
| **Friction** | 1.0 ps⁻¹ | Langevin friction coefficient |

### Input Files

| File | Description |
|------|-------------|
| `reference.pdb` | Defines protein topology and atom ordering |
| `start_positions.npy` | Initial coordinates for 100 replicas (shape: `100 × n_atoms × 3`) |

---
This simulation takes ~12 hours on an NVIDIA RTX 4080 GPU. To modify simulation parameters, see the full documentation: [`docs/commands/cg_sim.md`](../commands/cg_sim.md)


# CG Simulation Analysis

After the simulation is complete, run the following cells for some quick analysis!

In [ ]:
import numpy as np
import mdtraj as md
import h5py
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
cmap = mpl.colors.LinearSegmentedColormap.from_list("mycmap", ['#1B346C','#01ABE9','#F1F8F1','#DB3A4B'])

## CV Setup

In [33]:
pdb = md.load("trpcage/reference.pdb")
cg_indices = pdb.top.select("name CA or name C or name N")
cg_pdb = pdb.atom_slice(cg_indices)
num_cg_atoms = cg_pdb.n_atoms
ca_indices = cg_pdb.top.select("name CA")
dihedral_indices = [ca_indices[-4:]]
native_contact_indices = np.load("trpcage/native_contact_cg_indices.npy")

def get_trpcage_reaction_coordinates(traj, cutoff=6):
    assert traj.n_atoms == num_cg_atoms
    dihedrals = md.compute_dihedrals(traj, dihedral_indices).reshape(-1)
    contacts = md.compute_distances(traj, native_contact_indices) * 10
    contacts = (cutoff - contacts + 0.75)
    contacts = (1 / (1 + np.exp( - 2 * contacts))).mean(axis=1)
    return np.stack([dihedrals, contacts], axis=1)

# CG Simulations initialized to starting_positions
starting_positions = np.load("trpcage/start_positions.npy")[:, cg_indices, :] / 10
starting_cv = get_trpcage_reaction_coordinates(md.Trajectory(starting_positions, cg_pdb.top))

## Collecting CG Simulation Data

In [ ]:
cg_sim_data_folder = "/path/to/sim" # Replace with actual path

h5_file_paths = [f"{cg_sim_data_folder}/{x}" for x in os.listdir(cg_sim_data_folder) if "hdf5" in x]
chk_files = [f"{cg_sim_data_folder}/{x}" for x in os.listdir(cg_sim_data_folder) if "chk" in x]
assert len(h5_file_paths) == len(chk_files), "Number of hdf5 files and chk files should be the same"

h5_file_paths = sorted(h5_file_paths, key=lambda x: int(x.split("_")[-1].split(".")[0]))
burn_in = 1 # Number of chk files to skip for burn-in
h5_file_paths = h5_file_paths[burn_in:]
positions = np.concatenate([h5py.File(y, "r")["positions"][:, :, :, :] for y in h5_file_paths], axis=0)
batch_size = positions.shape[1]

# Check for NaN values in each simulation, if any NaN values are found, print the indices of the simulations that contain them
print(np.where(np.isnan(positions).any(axis=(0, 2, 3))))

positions = positions.reshape(-1, num_cg_atoms, 3)

# Analysis

In [ ]:
cv = get_trpcage_reaction_coordinates(md.Trajectory(positions / 10, cg_pdb.top))
init_z, init_rc1_edge, init_rc2_edge = np.histogram2d(cv[:, 0],
                                                     cv[:, 1],
                                                   #   range=[[-2.5, 2.5], [-2, 4]],
                                                        bins=20)


init_rc1 = 0.5 * (init_rc1_edge[:-1] + init_rc1_edge[1:])
init_rc2 = 0.5 * (init_rc2_edge[:-1] + init_rc2_edge[1:])
init_z = init_z.T
init_z_density = init_z/float(init_z.sum())
init_free_energy = np.inf * np.ones(shape=init_z.shape)

nonzero = init_z_density.nonzero()
init_free_energy[nonzero] = -np.log(init_z_density[nonzero])
init_free_energy = init_free_energy - init_free_energy.min()


plt.figure(figsize=(5, 4))
vmax = 7
plt.contourf(init_rc1, init_rc2, init_free_energy,
                levels=np.linspace(0, vmax, vmax+4), cmap=cmap)
cbar = plt.colorbar(boundaries=np.linspace(0, vmax, vmax+4), pad=0.01)
cbar.set_label("Free Energy (kT)", rotation=270, labelpad=20, fontsize=16)
cbar.set_ticks(np.arange(0, vmax + 1, 1))

# make X and Y axis ticks every 0.5
plt.gca().xaxis.set_major_locator(mpl.ticker.MultipleLocator(1.0))
plt.gca().yaxis.set_major_locator(mpl.ticker.MultipleLocator(0.2))

plt.xlabel("Native Contacts", fontsize=16)
plt.ylabel("Dihedral", fontsize=16)
plt.xlim(-np.pi, np.pi)
plt.ylim(0.15, .8)
plt.tight_layout()

In [ ]:
# To view the path of each trajectory
for index in range(batch_size):
    plt.contourf(init_rc1, init_rc2, init_free_energy,
                    levels=np.linspace(0, vmax, vmax+4), cmap=cmap)
    cbar = plt.colorbar(boundaries=np.linspace(0, vmax, vmax+4), pad=0.01)
    cbar.set_label(r"Free Energy", rotation=270, labelpad=20)
    index_cv = cv[index::batch_size, :]
    
    plt.title(f"Trajectory {index}")
    plt.plot(index_cv[::10, 0], index_cv[::10, 1], color='red')
    plt.scatter(index_cv[-1, 0], index_cv[-1, 1], color='black', s=20, label="End", zorder=10)
    plt.scatter(starting_cv[index, 0], starting_cv[index, 1], color='blue', s=20, label="Start", zorder=10)
    plt.legend()
    plt.xlim(-np.pi, np.pi)
    plt.show()